In [11]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from category_encoders import TargetEncoder

In [12]:
# =========================================================
# Config
# =========================================================

PROJECT_ROOT = Path("/mnt/c/dev/my_ml_project")

DATA_DIR = PROJECT_ROOT / "data"
OOF_DIR = PROJECT_ROOT / "oof_preds"
SUB_DIR = PROJECT_ROOT / "submissions"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

TARGET = "임신 성공 여부"

FOLD_SEED = 42
N_SPLITS = 5
POS_WEIGHT = 190123 / 66228

BASELINE_SEED42_FINAL_OOF = 0.7404963277840849

SAVE_DIR = OOF_DIR / "combo_te_v1_transfer_day_seed42"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

SUB_SAVE_DIR = SUB_DIR / "transfer_day"
SUB_SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PATH exists:", TRAIN_PATH.exists())
print("SAVE_DIR:", SAVE_DIR)
print("SUB_SAVE_DIR:", SUB_SAVE_DIR)

PROJECT_ROOT: /mnt/c/dev/my_ml_project
TRAIN_PATH exists: True
SAVE_DIR: /mnt/c/dev/my_ml_project/oof_preds/combo_te_v1_transfer_day_seed42
SUB_SAVE_DIR: /mnt/c/dev/my_ml_project/submissions/transfer_day


In [13]:
def data_preprocessing(df):
    df = df.copy()
    time_cols = [
        '임신 시도 또는 마지막 임신 경과 연수',
        '난자 해동 경과일',
        '난자 혼합 경과일',
        '배아 이식 경과일',
        '배아 해동 경과일'
    ]

    for col in time_cols:
        df[f'{col}_performed'] = (
            df[col].notnull()
        ).astype(int)


    df = df.fillna(0)
    infertility_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 남성 요인',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    male_cols = [
        '불임 원인 - 남성 요인',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    female_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증'
    ]

    df['불임원인_총개수'] = df[infertility_cols].sum(axis=1)

    df['남성_원인_수'] = df[male_cols].sum(axis=1)

    df['여성_원인_수'] = df[female_cols].sum(axis=1)

    df['남녀_복합_원인'] = (
        (df['남성_원인_수'] > 0) &
        (df['여성_원인_수'] > 0)
    ).astype(int)

    df['원인불명'] = (
        df['불임원인_총개수'] == 0
    ).astype(int)

    count_cols = [
        '총 시술 횟수',
        'IVF 시술 횟수',
        'DI 시술 횟수',
        '총 임신 횟수',
        'IVF 임신 횟수',
        'DI 임신 횟수',
        '총 출산 횟수',
        'IVF 출산 횟수',
        'DI 출산 횟수',
        '클리닉 내 총 시술 횟수'
    ]

    count_map = {
        '0회': 0,
        '1회': 1,
        '2회': 2,
        '3회': 3,
        '4회': 4,
        '5회': 5,
        '6회 이상': 6,
    }

    for col in count_cols:
        df[col] = df[col].map(count_map).astype(float)

    df['고령여부'] = df['시술 당시 나이'].isin([
        '만38-39세',
        '만40-42세',
        '만43-44세',
        '만45-50세'
    ]).astype(int)


    df['배아_생성률'] = np.where(
        df['혼합된 난자 수'] == 0,
        0,
        df['총 생성 배아 수'] / df['혼합된 난자 수']
    )

    # 2. 배아 이식 효율
    df['배아_이식률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['이식된 배아 수'] / df['총 생성 배아 수']
    )

    # 3. 배아 냉동 비율
    df['배아_냉동률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['저장된 배아 수'] / df['총 생성 배아 수']
    )
    df['IVF_임신성공률'] = np.where(
        df['IVF 시술 횟수'] == 0,
        0,
        df['IVF 임신 횟수'] / df['IVF 시술 횟수']
    )

    df['DI_임신성공률'] = np.where(
        df['DI 시술 횟수'] == 0,
        0,
        df['DI 임신 횟수'] / df['DI 시술 횟수']
    )

    df['고령_난자수_interaction'] = (
        df['고령여부'] *
        df['수집된 신선 난자 수']
    )

    df['배아이식_수행여부'] = (
        df['이식된 배아 수'] > 0
    ).astype(int)

    df['배아_이식_집중도'] = np.where(
        (df['이식된 배아 수'] + df['저장된 배아 수']) == 0,
        0,
        df['이식된 배아 수'] /
        (
            df['이식된 배아 수'] +
            df['저장된 배아 수']
        )
    )
    df["배아 생성 주요 이유"] = (
        df["배아 생성 주요 이유"]
        .astype(str)
        .astype("category")
    )
    df["특정 시술 유형"] = (
        df["특정 시술 유형"]
        .astype(str)
        .astype("category")
    )

    # 고령 × 이식 배아 수
    df['고령_배아이식'] = (
        df['고령여부'] *
        df['이식된 배아 수']
    )

    # 고령 × 총 생성 배아 수
    df['고령_배아생성'] = (
        df['고령여부'] *
        df['총 생성 배아 수']
    )

    # 고령 × 저장 배아 수
    df['고령_배아저장'] = (
        df['고령여부'] *
        df['저장된 배아 수']
    )

    # 고령 × 미세주입 난자 수
    df['고령_미세주입난자'] = (
        df['고령여부'] *
        df['미세주입된 난자 수']
    )

    df['출산_임신_전환율'] = np.where(
        df['총 임신 횟수'] == 0,
        0,
        df['총 출산 횟수'] / df['총 임신 횟수']
    )

    df['클리닉_집중도'] = np.where(
        df['총 시술 횟수'] == 0,
        0,
        df['클리닉 내 총 시술 횟수'] / df['총 시술 횟수']
    )

    df['첫_시술_여부'] = (
        df['총 시술 횟수'] == 0
    ).astype(int)



    binary_keywords = [
        "코드", "나이", "유형", "여부", "원인", "이유", "횟수", "출처"
    ]

    binary_cols = [
        col for col in df.columns
        if any(keyword in col for keyword in binary_keywords)
    ]

    df[binary_cols] = df[binary_cols].astype('category')

    object_cols = df.select_dtypes(include="object").columns

    df[object_cols] = df[object_cols].astype(str)

    cat_cols = df.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for col in cat_cols:
        if col != TARGET:
            df[col] = df[col].astype(str)

    drop_cols = [
        '배아이식_수행여부'
    ]
    df = df.drop(
        columns=drop_cols
        )
    return df

In [14]:
def _num(df, col, fill_value=0):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").fillna(fill_value)
    return pd.Series(fill_value, index=df.index)


def add_transfer_day_features(df):
    """
    OOF slice 분석 기반 targeted feature.

    추가 feature:
    - 배아이식일_0_flag
    - 배아이식일_1_flag
    - 배아이식일_5_flag
    - 배아이식일_0_or_1
    - 배아이식일_4_or_5
    - 이식일0_이식배아수
    - 이식일1_이식배아수
    - 이식일5_이식배아수
    """
    df = df.copy()

    transfer_embryo = _num(df, "이식된 배아 수")

    if "배아 이식 경과일" in df.columns:
        transfer_day = pd.to_numeric(df["배아 이식 경과일"], errors="coerce")
    else:
        transfer_day = pd.Series(np.nan, index=df.index)

    df["배아이식일_0_flag"] = (transfer_day == 0).astype(int)
    df["배아이식일_1_flag"] = (transfer_day == 1).astype(int)
    df["배아이식일_5_flag"] = (transfer_day == 5).astype(int)

    df["배아이식일_0_or_1"] = (
        transfer_day.isin([0, 1])
    ).astype(int)

    df["배아이식일_4_or_5"] = (
        transfer_day.isin([4, 5])
    ).astype(int)

    df["이식일0_이식배아수"] = (
        df["배아이식일_0_flag"] * transfer_embryo
    )

    df["이식일1_이식배아수"] = (
        df["배아이식일_1_flag"] * transfer_embryo
    )

    df["이식일5_이식배아수"] = (
        df["배아이식일_5_flag"] * transfer_embryo
    )

    return df


def data_preprocessing_transfer_day(df):
    df = df.copy()

    # raw 상태에서 transfer_day feature 추가
    df = add_transfer_day_features(df)

    # 기존 champion preprocessing 적용
    df = data_preprocessing(df)

    return df

In [15]:
def rank01(pred):
    return rankdata(pred) / len(pred)


def add_combo_columns(X):
    X = X.copy()

    combo_pairs = [
        ("시술 당시 나이", "난자 출처"),
        ("시술 당시 나이", "정자 출처"),
        ("시술 당시 나이", "시술 유형"),
        ("시술 당시 나이", "특정 시술 유형"),
        ("시술 유형", "난자 출처"),
        ("시술 유형", "정자 출처"),
        ("특정 시술 유형", "난자 출처"),
        ("특정 시술 유형", "정자 출처"),
        ("난자 출처", "정자 출처"),
        ("배란 유도 유형", "시술 당시 나이"),
    ]

    combo_cols = []

    for col1, col2 in combo_pairs:
        if col1 in X.columns and col2 in X.columns:
            new_col = f"{col1}_{col2}_combo"
            X[new_col] = X[col1].astype(str) + "_" + X[col2].astype(str)
            combo_cols.append(new_col)

    return X, combo_cols


def add_oof_target_encoding(X, y, cols, n_splits=5, smoothing=10, random_state=42):
    X = X.copy().reset_index(drop=True)
    y = y.astype(int).reset_index(drop=True)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    te_features = pd.DataFrame(index=X.index)

    for col in cols:
        te_features[f"{col}_TE"] = 0.0

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f"Target Encoding Fold {fold}")

        X_tr = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_tr = y.iloc[train_idx]

        encoder = TargetEncoder(
            cols=cols,
            smoothing=smoothing
        )

        encoder.fit(X_tr[cols], y_tr)
        encoded_val = encoder.transform(X_val[cols])

        for col in cols:
            te_features.loc[val_idx, f"{col}_TE"] = encoded_val[col].values

    final_encoder = TargetEncoder(
        cols=cols,
        smoothing=smoothing
    )

    final_encoder.fit(X[cols], y)

    X_te = pd.concat([X, te_features], axis=1)

    return X_te, final_encoder


def make_cat_params(kind="main", seed=42):
    if kind == "main":
        return dict(
            iterations=2000,
            learning_rate=0.02498214961001344,
            depth=8,
            l2_leaf_reg=18.591182129683194,
            random_strength=0.32969640414889206,
            bagging_temperature=4.535604806522509,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=100,
            class_weights=[1, POS_WEIGHT],
            allow_writing_files=False,
        )

    if kind == "shallow":
        return dict(
            iterations=2500,
            learning_rate=0.02,
            depth=5,
            l2_leaf_reg=8,
            random_strength=1.5,
            bagging_temperature=2,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=100,
            class_weights=[1, POS_WEIGHT],
            allow_writing_files=False,
        )

    if kind == "random":
        return dict(
            iterations=2500,
            learning_rate=0.022,
            depth=7,
            l2_leaf_reg=15,
            random_strength=8,
            bagging_temperature=8,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=100,
            class_weights=[1, POS_WEIGHT],
            allow_writing_files=False,
        )

    raise ValueError(f"Unknown kind: {kind}")

In [16]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
submission = pd.read_csv(SUBMISSION_PATH)

X_raw = train_raw.drop(columns=[TARGET])
y = train_raw[TARGET].astype(int).reset_index(drop=True)
X_test_raw = test_raw.copy()

id_cols = [col for col in X_raw.columns if "ID" in col.upper()]
X_raw = X_raw.drop(columns=id_cols, errors="ignore")
X_test_raw = X_test_raw.drop(columns=id_cols, errors="ignore")

# transfer_day feature 포함 preprocessing
X = data_preprocessing_transfer_day(X_raw)
X_test = data_preprocessing_transfer_day(X_test_raw)

transfer_day_cols = [
    "배아이식일_0_flag",
    "배아이식일_1_flag",
    "배아이식일_5_flag",
    "배아이식일_0_or_1",
    "배아이식일_4_or_5",
    "이식일0_이식배아수",
    "이식일1_이식배아수",
    "이식일5_이식배아수",
]

print("transfer_day cols check")
print(X[transfer_day_cols].nunique())
display(X[transfer_day_cols].head())

X_combo, combo_cols = add_combo_columns(X)
X_test_combo, _ = add_combo_columns(X_test)

base_te_cols = [
    "시술 시기 코드",
    "시술 유형",
    "특정 시술 유형",
    "배란 유도 유형",
    "난자 출처",
    "정자 출처",
    "배아 생성 주요 이유",
    "시술 당시 나이",
]

# 첫 OOF에서는 transfer_day feature를 TE에 추가하지 않음
base_te_cols = [col for col in base_te_cols if col in X_combo.columns]
te_cols = base_te_cols + combo_cols

print("base_te_cols:", base_te_cols)
print("combo_cols:", combo_cols)
print("te_cols count:", len(te_cols))

X_te_combo, te_encoder_combo = add_oof_target_encoding(
    X_combo,
    y,
    cols=te_cols,
    n_splits=N_SPLITS,
    smoothing=10,
    random_state=FOLD_SEED
)

test_te_values = te_encoder_combo.transform(X_test_combo[te_cols])

for col in te_cols:
    X_test_combo[f"{col}_TE"] = test_te_values[col].values

X_te_combo = X_te_combo.drop(columns=combo_cols, errors="ignore")
X_test_te_combo = X_test_combo.drop(columns=combo_cols, errors="ignore")

X_test_te_combo = X_test_te_combo[X_te_combo.columns]

X_stack = X_te_combo.copy().reset_index(drop=True)
X_test_stack = X_test_te_combo.copy().reset_index(drop=True)
y_stack = y.reset_index(drop=True)

cat_cols = X_stack.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in cat_cols:
    X_stack[col] = X_stack[col].astype(str)
    X_test_stack[col] = X_test_stack[col].astype(str)

print("X_stack:", X_stack.shape)
print("X_test_stack:", X_test_stack.shape)
print("cat_cols:", len(cat_cols))
print("combo cols remaining:", [c for c in X_stack.columns if c.endswith("_combo")])

assert list(X_stack.columns) == list(X_test_stack.columns)
assert len(X_stack) == len(y_stack)
assert len(X_test_stack) == len(submission)

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=FOLD_SEED
)

transfer_day cols check
배아이식일_0_flag    2
배아이식일_1_flag    2
배아이식일_5_flag    2
배아이식일_0_or_1    2
배아이식일_4_or_5    2
이식일0_이식배아수      4
이식일1_이식배아수      4
이식일5_이식배아수      4
dtype: int64


,배아이식일_0_flag,배아이식일_1_flag,배아이식일_5_flag,배아이식일_0_or_1,배아이식일_4_or_5,이식일0_이식배아수,이식일1_이식배아수,이식일5_이식배아수
0,0,0,0,0,0,0.0,0.0,0.0
1,0,0,0,0,0,0.0,0.0,0.0
2,0,0,0,0,0,0.0,0.0,0.0
3,0,0,0,0,0,0.0,0.0,0.0
4,0,0,0,0,0,0.0,0.0,0.0


base_te_cols: ['시술 시기 코드', '시술 유형', '특정 시술 유형', '배란 유도 유형', '난자 출처', '정자 출처', '배아 생성 주요 이유', '시술 당시 나이']
combo_cols: ['시술 당시 나이_난자 출처_combo', '시술 당시 나이_정자 출처_combo', '시술 당시 나이_시술 유형_combo', '시술 당시 나이_특정 시술 유형_combo', '시술 유형_난자 출처_combo', '시술 유형_정자 출처_combo', '특정 시술 유형_난자 출처_combo', '특정 시술 유형_정자 출처_combo', '난자 출처_정자 출처_combo', '배란 유도 유형_시술 당시 나이_combo']
te_cols count: 18
Target Encoding Fold 1
Target Encoding Fold 2
Target Encoding Fold 3
Target Encoding Fold 4
Target Encoding Fold 5
X_stack: (256351, 118)
X_test_stack: (90067, 118)
cat_cols: 54
combo cols remaining: []


In [17]:
def train_cat_seed_ensemble(
    X_stack,
    X_test_stack,
    y_stack,
    cat_cols,
    skf,
    seeds,
    kind="main",
    name="main_cat"
):
    seed_oof_list = []
    seed_test_list = []
    score_rows = []

    for seed in seeds:
        print(f"\n================ {name} seed {seed} ================")

        oof = np.zeros(len(X_stack))
        test_pred = np.zeros(len(X_test_stack))

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
            print(f"\n{name} seed {seed} / Fold {fold}")

            X_tr = X_stack.iloc[tr_idx].copy()
            X_val = X_stack.iloc[val_idx].copy()
            y_tr = y_stack.iloc[tr_idx]
            y_val = y_stack.iloc[val_idx]

            model = CatBoostClassifier(**make_cat_params(kind=kind, seed=seed))

            early_stop = 100 if kind == "main" else 150

            model.fit(
                X_tr,
                y_tr,
                cat_features=cat_cols,
                eval_set=(X_val, y_val),
                early_stopping_rounds=early_stop,
                verbose=100
            )

            val_pred = model.predict_proba(X_val)[:, 1]
            oof[val_idx] = val_pred

            fold_auc = roc_auc_score(y_val, val_pred)
            print(f"{name} seed {seed} Fold {fold} AUC:", fold_auc)

            score_rows.append({
                "model": name,
                "seed": seed,
                "fold": fold,
                "auc": fold_auc
            })

            test_pred += model.predict_proba(X_test_stack)[:, 1] / skf.n_splits

        seed_auc = roc_auc_score(y_stack, oof)
        print(f"\n{name} seed {seed} OOF AUC:", seed_auc)

        score_rows.append({
            "model": name,
            "seed": seed,
            "fold": "OOF",
            "auc": seed_auc
        })

        seed_oof_list.append(oof)
        seed_test_list.append(test_pred)

        np.save(SAVE_DIR / f"{name}_seed{seed}_oof.npy", oof)
        np.save(SAVE_DIR / f"{name}_seed{seed}_test.npy", test_pred)

    final_oof = np.mean(seed_oof_list, axis=0)
    final_test = np.mean(seed_test_list, axis=0)

    final_auc = roc_auc_score(y_stack, final_oof)
    print(f"\n{name} seed ensemble OOF AUC:", final_auc)

    score_rows.append({
        "model": name,
        "seed": "ensemble",
        "fold": "OOF",
        "auc": final_auc
    })

    return final_oof, final_test, pd.DataFrame(score_rows)


def train_cat_single(
    X_stack,
    X_test_stack,
    y_stack,
    cat_cols,
    skf,
    kind="shallow",
    name="shallow_cat"
):
    oof = np.zeros(len(X_stack))
    test_pred = np.zeros(len(X_test_stack))
    score_rows = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
        print(f"\n================ {name} Fold {fold} ================")

        X_tr = X_stack.iloc[tr_idx].copy()
        X_val = X_stack.iloc[val_idx].copy()
        y_tr = y_stack.iloc[tr_idx]
        y_val = y_stack.iloc[val_idx]

        model = CatBoostClassifier(**make_cat_params(kind=kind, seed=42))

        early_stop = 150 if kind == "shallow" else 200

        model.fit(
            X_tr,
            y_tr,
            cat_features=cat_cols,
            eval_set=(X_val, y_val),
            early_stopping_rounds=early_stop,
            verbose=100
        )

        val_pred = model.predict_proba(X_val)[:, 1]
        oof[val_idx] = val_pred

        fold_auc = roc_auc_score(y_val, val_pred)
        print(f"{name} Fold {fold} AUC:", fold_auc)

        score_rows.append({
            "model": name,
            "seed": 42,
            "fold": fold,
            "auc": fold_auc
        })

        test_pred += model.predict_proba(X_test_stack)[:, 1] / skf.n_splits

        np.save(SAVE_DIR / f"{name}_partial_oof_fold{fold}.npy", oof)
        np.save(SAVE_DIR / f"{name}_partial_test_fold{fold}.npy", test_pred)

    final_auc = roc_auc_score(y_stack, oof)
    print(f"\n{name} OOF AUC:", final_auc)

    score_rows.append({
        "model": name,
        "seed": 42,
        "fold": "OOF",
        "auc": final_auc
    })

    return oof, test_pred, pd.DataFrame(score_rows)

In [18]:
main_cat_oof, main_cat_test_pred, main_score_df = train_cat_seed_ensemble(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    cat_cols=cat_cols,
    skf=skf,
    seeds=[42, 77, 2024],
    kind="main",
    name="main_cat"
)

np.save(SAVE_DIR / "main_cat_oof.npy", main_cat_oof)
np.save(SAVE_DIR / "main_cat_test_pred.npy", main_cat_test_pred)

shallow_cat_oof, shallow_cat_test_pred, shallow_score_df = train_cat_single(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    cat_cols=cat_cols,
    skf=skf,
    kind="shallow",
    name="shallow_cat"
)

np.save(SAVE_DIR / "shallow_cat_oof.npy", shallow_cat_oof)
np.save(SAVE_DIR / "shallow_cat_test_pred.npy", shallow_cat_test_pred)

random_cat_oof, random_cat_test_pred, random_score_df = train_cat_single(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    cat_cols=cat_cols,
    skf=skf,
    kind="random",
    name="random_cat"
)

np.save(SAVE_DIR / "random_cat_oof.npy", random_cat_oof)
np.save(SAVE_DIR / "random_cat_test_pred.npy", random_cat_test_pred)

def train_xgb_oof_test(X_stack, X_test_stack, y_stack, skf):
    numeric_features = X_stack.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X_stack.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ])

    oof = np.zeros(len(X_stack))
    test_pred = np.zeros(len(X_test_stack))
    score_rows = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
        print(f"\n================ XGB Fold {fold} ================")

        X_tr = X_stack.iloc[tr_idx].copy()
        X_val = X_stack.iloc[val_idx].copy()
        y_tr = y_stack.iloc[tr_idx]
        y_val = y_stack.iloc[val_idx]

        X_tr_trans = preprocessor.fit_transform(X_tr)
        X_val_trans = preprocessor.transform(X_val)
        X_test_trans = preprocessor.transform(X_test_stack)

        model = XGBClassifier(
            n_estimators=1000,
            learning_rate=0.03,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="auc",
            random_state=42,
            scale_pos_weight=POS_WEIGHT,
            tree_method="hist",
            n_jobs=-1
        )

        model.fit(
            X_tr_trans,
            y_tr.to_numpy().ravel(),
            eval_set=[(X_val_trans, y_val.to_numpy().ravel())],
            verbose=False
        )

        val_pred = model.predict_proba(X_val_trans)[:, 1]
        oof[val_idx] = val_pred

        fold_auc = roc_auc_score(y_val, val_pred)
        print("XGB Fold AUC:", fold_auc)

        score_rows.append({
            "model": "xgb",
            "seed": 42,
            "fold": fold,
            "auc": fold_auc
        })

        test_pred += model.predict_proba(X_test_trans)[:, 1] / skf.n_splits

        np.save(SAVE_DIR / f"xgb_partial_oof_fold{fold}.npy", oof)
        np.save(SAVE_DIR / f"xgb_partial_test_fold{fold}.npy", test_pred)

    final_auc = roc_auc_score(y_stack, oof)
    print("\nXGB OOF AUC:", final_auc)

    score_rows.append({
        "model": "xgb",
        "seed": 42,
        "fold": "OOF",
        "auc": final_auc
    })

    return oof, test_pred, pd.DataFrame(score_rows)


xgb_oof, xgb_test_pred, xgb_score_df = train_xgb_oof_test(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    skf=skf
)

np.save(SAVE_DIR / "xgb_oof.npy", xgb_oof)
np.save(SAVE_DIR / "xgb_test_pred.npy", xgb_test_pred)


================ main_cat seed 42 ================

main_cat seed 42 / Fold 1
0:	test: 0.7278284	best: 0.7278284 (0)	total: 289ms	remaining: 9m 36s
100:	test: 0.7361582	best: 0.7361582 (100)	total: 15.5s	remaining: 4m 51s
200:	test: 0.7374208	best: 0.7374238 (198)	total: 29s	remaining: 4m 19s
300:	test: 0.7377645	best: 0.7377965 (281)	total: 42.2s	remaining: 3m 57s
400:	test: 0.7379148	best: 0.7379148 (400)	total: 55s	remaining: 3m 39s
500:	test: 0.7379753	best: 0.7379783 (470)	total: 1m 8s	remaining: 3m 24s
600:	test: 0.7380033	best: 0.7380216 (589)	total: 1m 23s	remaining: 3m 13s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7380215729
bestIteration = 589

Shrink model to first 590 iterations.
main_cat seed 42 Fold 1 AUC: 0.7380215728604211

main_cat seed 42 / Fold 2
0:	test: 0.7273076	best: 0.7273076 (0)	total: 185ms	remaining: 6m 9s
100:	test: 0.7386656	best: 0.7386656 (100)	total: 15.6s	remaining: 4m 54s
200:	test: 0.7412133	best: 0.7412133 (200)	total: 29.

In [20]:
main_cat_rank_oof = rank01(main_cat_oof)
shallow_cat_rank_oof = rank01(shallow_cat_oof)
random_cat_rank_oof = rank01(random_cat_oof)
xgb_rank_oof = rank01(xgb_oof)

final_oof_transfer_day_seed42 = (
    0.44 * main_cat_rank_oof +
    0.08 * shallow_cat_rank_oof +
    0.35 * random_cat_rank_oof +
    0.13 * xgb_rank_oof
)

final_oof_auc = roc_auc_score(y_stack, final_oof_transfer_day_seed42)

print("\n====================")
print("Transfer Day Seed42 Main Cat OOF:", roc_auc_score(y_stack, main_cat_oof))
print("Transfer Day Seed42 Shallow Cat OOF:", roc_auc_score(y_stack, shallow_cat_oof))
print("Transfer Day Seed42 Random Cat OOF:", roc_auc_score(y_stack, random_cat_oof))
print("Transfer Day Seed42 XGB OOF:", roc_auc_score(y_stack, xgb_oof))
print("Transfer Day Seed42 Final Rank Blend OOF:", final_oof_auc)

print("\nBaseline seed42 final OOF:", BASELINE_SEED42_FINAL_OOF)
print("diff:", final_oof_auc - BASELINE_SEED42_FINAL_OOF)

np.save(SAVE_DIR / "final_oof_transfer_day_seed42.npy", final_oof_transfer_day_seed42)
np.save(SAVE_DIR / "y_stack.npy", y_stack.to_numpy())

score_df = pd.concat(
    [main_score_df, shallow_score_df, random_score_df, xgb_score_df],
    ignore_index=True
)

score_df.to_csv(SAVE_DIR / "fold_oof_scores.csv", index=False)

summary_df = pd.DataFrame([
    {
        "fold_seed": FOLD_SEED,
        "feature_set": "combo_te_v1_transfer_day",
        "main_cat_oof": roc_auc_score(y_stack, main_cat_oof),
        "shallow_cat_oof": roc_auc_score(y_stack, shallow_cat_oof),
        "random_cat_oof": roc_auc_score(y_stack, random_cat_oof),
        "xgb_oof": roc_auc_score(y_stack, xgb_oof),
        "final_rank_blend_oof": final_oof_auc,
        "baseline_seed42_oof": BASELINE_SEED42_FINAL_OOF,
        "diff": final_oof_auc - BASELINE_SEED42_FINAL_OOF,
        "weights": "main 0.44 / shallow 0.08 / random 0.35 / xgb 0.13",
        "note": "transfer_day targeted features only. Not added to TE cols."
    }
])

summary_df.to_csv(SAVE_DIR / "summary.csv", index=False)
display(summary_df)


Transfer Day Seed42 Main Cat OOF: 0.7403529231564039
Transfer Day Seed42 Shallow Cat OOF: 0.7401727929799753
Transfer Day Seed42 Random Cat OOF: 0.7402672063307199
Transfer Day Seed42 XGB OOF: 0.7380123004364589
Transfer Day Seed42 Final Rank Blend OOF: 0.7404438437049721

Baseline seed42 final OOF: 0.7404963277840849
diff: -5.248407911284669e-05


,fold_seed,feature_set,main_cat_oof,shallow_cat_oof,random_cat_oof,xgb_oof,final_rank_blend_oof,baseline_seed42_oof,diff,weights,note
0,42,combo_te_v1_transfer_day,0.740353,0.740173,0.740267,0.738012,0.740444,0.740496,-0.000052,main 0.44 / shallow 0.08 / random 0.35 / xgb 0.13,transfer_day targeted features only. Not added...


In [21]:
from pathlib import Path

SAVE_DIR = Path("/mnt/c/dev/my_ml_project/oof_preds/combo_te_v1_s10_seed2024")

for p in SAVE_DIR.glob("*fold2*.npy"):
    print(p.name)

random_cat_partial_oof_fold2.npy
random_cat_partial_test_fold2.npy
shallow_cat_partial_oof_fold2.npy
shallow_cat_partial_test_fold2.npy
xgb_partial_oof_fold2.npy
xgb_partial_test_fold2.npy


In [22]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

In [23]:
# =========================================================
# Config
# =========================================================

PROJECT_ROOT = Path("/mnt/c/dev/my_ml_project")

SUBMISSION_PATH = PROJECT_ROOT / "data" / "sample_submission.csv"

SAVE_DIR = PROJECT_ROOT / "oof_preds" / "fold2_probe"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

SUB_SAVE_DIR = PROJECT_ROOT / "submissions" / "fold2_probe"
SUB_SAVE_DIR.mkdir(parents=True, exist_ok=True)

TARGET_FOLD = 2
N_SPLITS = 5
FOLD_SEED = 42

print("SAVE_DIR:", SAVE_DIR)
print("SUB_SAVE_DIR:", SUB_SAVE_DIR)

SAVE_DIR: /mnt/c/dev/my_ml_project/oof_preds/fold2_probe
SUB_SAVE_DIR: /mnt/c/dev/my_ml_project/submissions/fold2_probe


In [24]:
def rank01(x):
    return rankdata(x) / len(x)


def train_fold2_cat_test_pred(
    X_stack,
    X_test_stack,
    y_stack,
    cat_cols,
    target_fold=2,
    kind="main",
    model_seed=42,
    fold_seed=42,
    name="main_seed42_fold2"
):
    """
    target_fold를 validation으로 두고,
    나머지 fold로 학습한 뒤 test prediction 반환.
    """
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=fold_seed
    )

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
        if fold != target_fold:
            continue

        print("\n" + "=" * 100)
        print(f"Training {name}")
        print(f"Train folds: all except fold {target_fold}")
        print(f"Valid fold: {target_fold}")
        print("=" * 100)

        X_tr = X_stack.iloc[tr_idx].copy()
        X_val = X_stack.iloc[val_idx].copy()
        y_tr = y_stack.iloc[tr_idx]
        y_val = y_stack.iloc[val_idx]

        model = CatBoostClassifier(
            **make_cat_params(kind=kind, seed=model_seed)
        )

        early_stop = 100 if kind == "main" else 150

        model.fit(
            X_tr,
            y_tr,
            cat_features=cat_cols,
            eval_set=(X_val, y_val),
            early_stopping_rounds=early_stop,
            verbose=100
        )

        val_pred = model.predict_proba(X_val)[:, 1]
        test_pred = model.predict_proba(X_test_stack)[:, 1]

        fold_auc = roc_auc_score(y_val, val_pred)

        print(f"{name} Fold {target_fold} AUC:", fold_auc)
        print("best_iteration:", model.best_iteration_)

        np.save(SAVE_DIR / f"{name}_val_pred.npy", val_pred)
        np.save(SAVE_DIR / f"{name}_test_pred.npy", test_pred)
        np.save(SAVE_DIR / f"{name}_val_idx.npy", val_idx)

        pd.DataFrame([{
            "name": name,
            "kind": kind,
            "model_seed": model_seed,
            "fold_seed": fold_seed,
            "target_fold": target_fold,
            "fold_auc": fold_auc,
            "best_iteration": model.best_iteration_,
        }]).to_csv(
            SAVE_DIR / f"{name}_summary.csv",
            index=False
        )

        return val_pred, test_pred, val_idx, fold_auc

    raise ValueError(f"target_fold={target_fold} not found")

In [25]:
fold2_results = {}

# main cat seed 42
val_pred, test_pred, val_idx, auc = train_fold2_cat_test_pred(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    cat_cols=cat_cols,
    target_fold=TARGET_FOLD,
    kind="main",
    model_seed=42,
    fold_seed=FOLD_SEED,
    name="main_cat_seed42_fold2"
)

fold2_results["main_cat_seed42_fold2"] = {
    "val_pred": val_pred,
    "test_pred": test_pred,
    "val_idx": val_idx,
    "auc": auc,
}


# main cat seed 2024
val_pred, test_pred, val_idx, auc = train_fold2_cat_test_pred(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    cat_cols=cat_cols,
    target_fold=TARGET_FOLD,
    kind="main",
    model_seed=2024,
    fold_seed=FOLD_SEED,
    name="main_cat_seed2024_fold2"
)

fold2_results["main_cat_seed2024_fold2"] = {
    "val_pred": val_pred,
    "test_pred": test_pred,
    "val_idx": val_idx,
    "auc": auc,
}


# random cat seed 42
val_pred, test_pred, val_idx, auc = train_fold2_cat_test_pred(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    cat_cols=cat_cols,
    target_fold=TARGET_FOLD,
    kind="random",
    model_seed=42,
    fold_seed=FOLD_SEED,
    name="random_cat_seed42_fold2"
)

fold2_results["random_cat_seed42_fold2"] = {
    "val_pred": val_pred,
    "test_pred": test_pred,
    "val_idx": val_idx,
    "auc": auc,
}


Training main_cat_seed42_fold2
Train folds: all except fold 2
Valid fold: 2
0:	test: 0.7273076	best: 0.7273076 (0)	total: 188ms	remaining: 6m 14s
100:	test: 0.7386656	best: 0.7386656 (100)	total: 40.8s	remaining: 12m 46s
200:	test: 0.7412133	best: 0.7412133 (200)	total: 55s	remaining: 8m 11s
300:	test: 0.7419709	best: 0.7419709 (300)	total: 1m 7s	remaining: 6m 22s
400:	test: 0.7423803	best: 0.7423822 (398)	total: 1m 19s	remaining: 5m 17s
500:	test: 0.7426325	best: 0.7426325 (500)	total: 1m 33s	remaining: 4m 38s
600:	test: 0.7428055	best: 0.7428105 (595)	total: 1m 47s	remaining: 4m 11s
700:	test: 0.7429748	best: 0.7429797 (696)	total: 2m 2s	remaining: 3m 46s
800:	test: 0.7430649	best: 0.7430856 (790)	total: 2m 17s	remaining: 3m 25s
900:	test: 0.7430888	best: 0.7430976 (892)	total: 2m 29s	remaining: 3m 1s
1000:	test: 0.7431087	best: 0.7431152 (919)	total: 2m 42s	remaining: 2m 41s
1100:	test: 0.7431435	best: 0.7431447 (1099)	total: 2m 56s	remaining: 2m 24s
1200:	test: 0.7431126	best: 0.7

In [26]:
for name, result in fold2_results.items():
    print(name, "fold2 auc:", result["auc"])

main_cat_seed42_fold2 fold2 auc: 0.7431599077617024
main_cat_seed2024_fold2 fold2 auc: 0.743286428406735
random_cat_seed42_fold2 fold2 auc: 0.7430988533750098


In [27]:
fold2_probe_test = (
    rank01(fold2_results["main_cat_seed42_fold2"]["test_pred"]) +
    rank01(fold2_results["main_cat_seed2024_fold2"]["test_pred"]) +
    rank01(fold2_results["random_cat_seed42_fold2"]["test_pred"])
) / 3

np.save(SAVE_DIR / "fold2_probe_test_rank_avg.npy", fold2_probe_test)

print("fold2_probe_test shape:", fold2_probe_test.shape)
print("min:", fold2_probe_test.min())
print("max:", fold2_probe_test.max())
print("mean:", fold2_probe_test.mean())

fold2_probe_test shape: (90067,)
min: 5.1813279743598274e-05
max: 0.9999814952572345
mean: 0.5000055514228297


In [28]:
submission = pd.read_csv(SUBMISSION_PATH)
pred_col = submission.columns[-1]

submission_fold2_only = submission.copy()
submission_fold2_only[pred_col] = fold2_probe_test

out_csv = SUB_SAVE_DIR / "submission_fold2_probe_only.csv"
out_npy = SUB_SAVE_DIR / "final_pred_fold2_probe_only.npy"

submission_fold2_only.to_csv(out_csv, index=False)
np.save(out_npy, fold2_probe_test)

print("saved:", out_csv)
print("saved:", out_npy)
print(submission_fold2_only[pred_col].describe())

saved: /mnt/c/dev/my_ml_project/submissions/fold2_probe/submission_fold2_probe_only.csv
saved: /mnt/c/dev/my_ml_project/submissions/fold2_probe/final_pred_fold2_probe_only.npy
count    90067.000000
mean         0.500006
std          0.288376
min          0.000052
25%          0.250016
50%          0.499809
75%          0.749722
max          0.999981
Name: probability, dtype: float64


In [29]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/mnt/c/dev/my_ml_project")

csv_path = PROJECT_ROOT / "submissions" / "fold2_probe" / "submission_fold2_probe_only.csv"
npy_path = PROJECT_ROOT / "submissions" / "fold2_probe" / "final_pred_fold2_probe_only.npy"
sample_path = PROJECT_ROOT / "data" / "sample_submission.csv"

print("csv exists:", csv_path.exists(), csv_path)
print("npy exists:", npy_path.exists(), npy_path)

sub = pd.read_csv(csv_path)
sample = pd.read_csv(sample_path)
pred = np.load(npy_path)

print("submission shape:", sub.shape)
print("sample shape:", sample.shape)
print("pred shape:", pred.shape)

pred_col = sub.columns[-1]

print(sub.head())
print(sub[pred_col].describe())

print("same columns:", list(sub.columns) == list(sample.columns))
print("same length:", len(sub) == len(sample))
print("finite:", np.isfinite(sub[pred_col]).all())
print("min >= 0:", sub[pred_col].min() >= 0)
print("max <= 1:", sub[pred_col].max() <= 1)
print("npy/csv same:", np.allclose(pred, sub[pred_col].values))

csv exists: True /mnt/c/dev/my_ml_project/submissions/fold2_probe/submission_fold2_probe_only.csv
npy exists: True /mnt/c/dev/my_ml_project/submissions/fold2_probe/final_pred_fold2_probe_only.npy
submission shape: (90067, 2)
sample shape: (90067, 2)
pred shape: (90067,)
           ID  probability
0  TEST_00000     0.054393
1  TEST_00001     0.134711
2  TEST_00002     0.265525
3  TEST_00003     0.205862
4  TEST_00004     0.933823
count    90067.000000
mean         0.500006
std          0.288376
min          0.000052
25%          0.250016
50%          0.499809
75%          0.749722
max          0.999981
Name: probability, dtype: float64
same columns: True
same length: True
finite: True
min >= 0: True
max <= 1: True
npy/csv same: True


In [30]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata

PROJECT_ROOT = Path("/mnt/c/dev/my_ml_project")

SUBMISSION_PATH = PROJECT_ROOT / "data" / "sample_submission.csv"

OUT_DIR = PROJECT_ROOT / "submissions" / "fold2_probe"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def rank01(x):
    return rankdata(x) / len(x)

# champion 4-seed
champion_path = (
    PROJECT_ROOT
    / "submissions"
    / "seed_ensemble"
    / "final_pred_combo_te_v1_seed42_2024_777_999_avg.npy"
)

# fold2 only
fold2_path = (
    PROJECT_ROOT
    / "submissions"
    / "fold2_probe"
    / "final_pred_fold2_probe_only.npy"
)

champion_pred = np.load(champion_path)
fold2_pred = np.load(fold2_path)

w_fold2 = 0.10

final_pred_w010 = (
    (1 - w_fold2) * rank01(champion_pred)
    + w_fold2 * rank01(fold2_pred)
)

submission = pd.read_csv(SUBMISSION_PATH)
pred_col = submission.columns[-1]

submission[pred_col] = final_pred_w010

out_csv = OUT_DIR / "submission_champion4seed_fold2probe_w0.10.csv"
out_npy = OUT_DIR / "final_pred_champion4seed_fold2probe_w0.10.npy"

submission.to_csv(out_csv, index=False)
np.save(out_npy, final_pred_w010)

print("saved csv:", out_csv)
print("saved npy:", out_npy)
print(submission[pred_col].describe())

saved csv: /mnt/c/dev/my_ml_project/submissions/fold2_probe/submission_champion4seed_fold2probe_w0.10.csv
saved npy: /mnt/c/dev/my_ml_project/submissions/fold2_probe/final_pred_champion4seed_fold2probe_w0.10.npy
count    90067.000000
mean         0.500006
std          0.288616
min          0.000012
25%          0.250177
50%          0.500037
75%          0.749984
max          0.999999
Name: probability, dtype: float64


In [31]:
check_sub = pd.read_csv(out_csv)
check_pred = np.load(out_npy)
sample = pd.read_csv(SUBMISSION_PATH)

print("csv exists:", out_csv.exists())
print("npy exists:", out_npy.exists())
print("same columns:", list(check_sub.columns) == list(sample.columns))
print("same length:", len(check_sub) == len(sample))
print("finite:", np.isfinite(check_sub[pred_col]).all())
print("min >= 0:", check_sub[pred_col].min() >= 0)
print("max <= 1:", check_sub[pred_col].max() <= 1)
print("npy/csv same:", np.allclose(check_pred, check_sub[pred_col].values))
print(check_sub.head())

csv exists: True
npy exists: True
same columns: True
same length: True
finite: True
min >= 0: True
max <= 1: True
npy/csv same: True
           ID  probability
0  TEST_00000     0.061976
1  TEST_00001     0.134991
2  TEST_00002     0.264653
3  TEST_00003     0.205813
4  TEST_00004     0.925098


In [32]:
import numpy as np
from pathlib import Path
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = Path("/mnt/c/dev/my_ml_project")
FOLD2_DIR = PROJECT_ROOT / "oof_preds" / "fold2_probe"

def rank01(x):
    return rankdata(x) / len(x)

# champion 4-seed OOF
seed42_oof = np.load(PROJECT_ROOT / "oof_preds" / "combo_te_s10" / "final_combo_rank.npy")
seed2024_oof = np.load(PROJECT_ROOT / "oof_preds" / "combo_te_v1_s10_seed2024" / "final_oof_seed2024.npy")
seed777_oof = np.load(PROJECT_ROOT / "oof_preds" / "combo_te_v1_s10_seed777" / "final_oof_seed777.npy")
seed999_oof = np.load(PROJECT_ROOT / "oof_preds" / "combo_te_v1_s10_seed999" / "final_oof_seed999.npy")

champion_oof = (seed42_oof + seed2024_oof + seed777_oof + seed999_oof) / 4

# fold2 validation 예측들
main42_val = np.load(FOLD2_DIR / "main_cat_seed42_fold2_val_pred.npy")
main2024_val = np.load(FOLD2_DIR / "main_cat_seed2024_fold2_val_pred.npy")
random42_val = np.load(FOLD2_DIR / "random_cat_seed42_fold2_val_pred.npy")

fold2_idx = np.load(FOLD2_DIR / "main_cat_seed42_fold2_val_idx.npy")

# fold2 probe validation rank avg
fold2_probe_val = (
    rank01(main42_val) +
    rank01(main2024_val) +
    rank01(random42_val)
) / 3

# fold2 구간의 y / champion pred
y_fold2 = y_stack.iloc[fold2_idx] if hasattr(y_stack, "iloc") else y_stack[fold2_idx]
champion_fold2 = champion_oof[fold2_idx]

print("Champion fold2 AUC:", roc_auc_score(y_fold2, champion_fold2))
print("Fold2 probe fold2 AUC:", roc_auc_score(y_fold2, fold2_probe_val))

Champion fold2 AUC: 0.7436109312955728
Fold2 probe fold2 AUC: 0.7434020315557035


In [33]:
champion_fold2_rank = rank01(champion_fold2)
fold2_probe_val_rank = rank01(fold2_probe_val)

for w in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    blend_fold2 = (
        (1 - w) * champion_fold2_rank +
        w * fold2_probe_val_rank
    )

    auc = roc_auc_score(y_fold2, blend_fold2)

    print(f"w_fold2={w:.2f} | fold2 AUC={auc:.10f}")

w_fold2=0.05 | fold2 AUC=0.7436245253
w_fold2=0.10 | fold2 AUC=0.7436333560
w_fold2=0.15 | fold2 AUC=0.7436409110
w_fold2=0.20 | fold2 AUC=0.7436491708
w_fold2=0.25 | fold2 AUC=0.7436508992
w_fold2=0.30 | fold2 AUC=0.7436523596
